# 15 · Profile: BLS Occupational Outlook Handbook, one-line job descriptions

**Method:** `docs/framework.md` §3. Card: `docs/source-cards/bls_ooh.md`. Plan: `docs/page3-build-plan.md` §4 item 1 (step 1 of §9).

**Why this source matters.** The job page (design brief §11.10 R2, option B) puts one plain sentence under the title: what the job is. BLS writes exactly that sentence for every Occupational Outlook Handbook (OOH) profile ("Accountants and auditors prepare and examine financial records."), in the third person, at a reading level meant for students. O\*NET's definitions (already staged) are SOC definitions in the imperative ("Examine, analyze, and interpret accounting records…"), so they're only the fallback for jobs the OOH doesn't cover.

**The catch.** One OOH profile can cover several SOC jobs, and its sentence is written for the group. For a job that shares a profile, the sentence can name a different job (33-3031 Fish and Game Wardens sits in "Police and Detectives": "Police officers protect lives and property…"). So every named job that shares a profile is read by hand (§5), and each verdict is a row in the seed `ref/ref_job_description_override.csv`, so it can be seen and reversed.

**Questions**
1. What is one row, and does each SOC code sit in exactly one profile?
2. How many of the 202 named job-page jobs get a BLS sentence, and how many through a shared profile?
3. For the shared ones, does the sentence read right under the job's own title? If not, which BLS sentence does?
4. What do the jobs without a profile fall back to?

In [1]:
import re
import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
sys.path.insert(0, str(PROJECT / "src"))
from common import RAW
from explore import q
from job_text import check_description_overrides, first_sentence, parse_ooh

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 30, "display.max_colwidth", 140, "display.width", 250, "display.max_rows", 120)
md = lambda s: display(Markdown(s))

man = pd.read_csv(RAW / "manifest.csv", dtype=str, keep_default_na=False)
FILE = man[man.key == "bls_ooh"].sort_values("downloaded_utc").iloc[-1]
display(FILE[["key", "file", "downloaded_utc", "bytes", "sha256"]].to_frame().T)
ooh = parse_ooh(RAW / FILE.file)
print(ooh["meta"])

,key,file,downloaded_utc,bytes,sha256
213,bls_ooh,skills/bls_ooh/xml-compilation_2026-10-16.xml,2026-10-16T16:43:25+00:00,13867897,6d661f1ac13c819b1170b48a18d4b0d4a16cf2687912c6c1ead56d50d414b894


{'publication_title': 'Occupational Outlook Handbook', 'current_year': '2025', 'projection_year': '2035', 'reference_year': '2025', 'reference_period': 'May 2025', 'last_updated': '8/26/2026'}


## 0. Source header

| | BLS Occupational Outlook Handbook, XML compilation (`bls_ooh`) |
|---|---|
| **Publisher** | U.S. Bureau of Labor Statistics, Office of Occupational Statistics and Employment Projections |
| **File** | `skills/bls_ooh/xml-compilation_2026-10-16.xml`, the whole handbook in one file ([developer page](https://www.bls.gov/ooh/about/ooh-developer-info.htm)). BLS overwrites it in place, so it's a dated snapshot |
| **Edition** | 2025–35 projections edition, wages for May 2025; file "last updated 8/26/2026" |
| **How it was made** | Written by BLS economists for each profile; SOC codes per profile from BLS. Not a survey: the sentence is editorial text |
| **License** | Public domain (U.S. government work); BLS asks for a citation |
| **Download** | bls.gov refuses a browser User-Agent; the project's descriptive one from `src/sources.py` works |

## 1. Load: one row per profile × SOC code

Each `<occupation>` is a profile. The parser is `src/job_text.py` (`parse_ooh`), shared with the load notebook 36. Kept: the profile code and title, the `<description>` sentence, the SOC codes under `<soc_coverage>`, the publish date, and two parts of the body used only in the hand check (§5): the "types of" paragraphs in *What They Do* (a bold job name followed by a sentence about that job) and the projections table in *Job Outlook* (one row per SOC code with BLS's title for it).

In [2]:
prof, socs, types, table = (ooh[k] for k in ["profiles", "socs", "types", "table"])
print(f"profiles {len(prof)} | profile x SOC rows {len(socs)} | 'types of' paragraphs {len(types)} in {types.ooh_code.nunique()} profiles"
      f" | projections-table rows {len(table)}")
display(prof.head(3))

profiles 343 | profile x SOC rows 606 | 'types of' paragraphs 738 in 182 profiles | projections-table rows 654


,ooh_code,ooh_title,description,description_raw,n_soc,publish_date
0,B047,Accountants and Auditors,Accountants and auditors prepare and examine financial records.,Accountants and auditors prepare and examine financial records.,1,2026-08-27
1,B151,Actors,"Actors express ideas and portray characters in theater, film, television, and other performing arts media.","Actors express ideas and portray characters in theater, film, television, and other performing arts media.",1,2026-08-27
2,B063,Actuaries,"Actuaries use mathematics, statistics, and financial theory to analyze the economic costs of risk and uncertainty.","Actuaries use mathematics, statistics, and financial theory to analyze the economic costs of risk and uncertainty.",1,2026-08-27


**Load findings**
- 343 profiles, 606 profile × SOC rows. 738 "types of" paragraphs in 182 profiles; the projections tables list 654 SOC rows (they also list the group total and, in a few profiles, codes outside the profile's coverage).

## 2. Grain

In [3]:
print("profile codes unique:", prof.ooh_code.is_unique, "| duplicated codes:", prof[prof.ooh_code.duplicated(keep=False)][["ooh_code", "ooh_title", "n_soc"]].to_dict("records"))
print("profiles with no SOC code:", prof[prof.n_soc == 0][["ooh_code", "ooh_title"]].to_dict("records"))
print("SOC codes in more than one profile:", socs.soc_code.duplicated().sum(), "| distinct SOC codes:", socs.soc_code.nunique())
print("SOC code format NN-NNNN:", socs.soc_code.str.fullmatch(r"\d\d-\d{4}").all(),
      "| ending in 0 (broad or BLS group codes):", sorted(socs.soc_code[socs.soc_code.str.endswith("0")]))
print("SOC codes per profile:", prof.n_soc.value_counts().sort_index().to_dict())
print("every coverage code is in the profile's projections table:",
      socs.merge(table, how="left", indicator=True)._merge.eq("both").mean().round(3))

profile codes unique: True | duplicated codes: []
profiles with no SOC code: [{'ooh_code': 'B389', 'ooh_title': 'Military Careers'}]
SOC codes in more than one profile: 0 | distinct SOC codes: 606
SOC code format NN-NNNN: True | ending in 0 (broad or BLS group codes): ['13-1020', '13-2020', '29-2010', '31-1120', '39-7010', '51-2090']
SOC codes per profile: {0: 1, 1: 244, 2: 52, 3: 15, 4: 16, 5: 6, 6: 2, 8: 2, 9: 1, 10: 1, 17: 1, 20: 1, 35: 1}
every coverage code is in the profile's projections table: 1.0


**Grain findings **
- One row = profile × SOC code; `soc_code` alone is unique (no SOC code sits in two profiles), so the staged table can be keyed on `soc_code`.
- One profile has no SOC code (*Military Careers*); it can't join to a job and drops out of staging.
- 244 profiles cover one SOC code; the rest cover 2 to 35 (*Postsecondary Teachers* has 35). Six codes are broad or BLS group codes (`13-1020`, `13-2020`, `29-2010`, `31-1120`, `39-7010`, `51-2090`), the same level `dim_job` uses for them.
- Every coverage code appears in its profile's projections table, so BLS's title for each code is available to the hand check.

## 3. Profile: the sentences and the dates

In [4]:
d = prof.description_raw.str.strip()
print("description length:", d.str.len().describe().round(0).to_dict())
print("sentences per description:", d.map(lambda s: len(re.findall(r"[a-z0-9)]\.(?:\s+[A-Z]|$)", s))).value_counts().to_dict())
print("HTML, entities or non-breaking spaces in the raw text:", d.str.contains(r"<|&\w+;|\xa0").sum(), "| repeats across profiles:", d.duplicated().sum())
print("publish dates:", prof.publish_date.value_counts().head(5).to_dict())
display(prof.loc[d.str.len().nlargest(3).index, ["ooh_title", "description"]])
print(prof.loc[d.str.contains(r"<|&\w+;|\xa0"), ["ooh_title", "description_raw", "description"]].to_string())

description length: {'count': 343.0, 'mean': 120.0, 'std': 38.0, 'min': 42.0, '25%': 96.0, '50%': 113.0, '75%': 141.0, 'max': 347.0}
sentences per description: {1: 315, 2: 25, 0: 2, 3: 1}
HTML, entities or non-breaking spaces in the raw text: 3 | repeats across profiles: 0
publish dates: {'2026-08-27': 343}


,ooh_title,description
217,Military Careers,Members of the U.S. military service train for and perform a variety of tasks in order to maintain the U.S. national defense. Servicemem...
80,Court Reporters and Simultaneous Captioners,"Court reporters create word-for-word transcriptions at trials, depositions, and other legal proceedings. Simultaneous captioners provide..."
61,"Compensation, Benefits, and Job Analysis Specialists","Compensation, benefits, and job analysis specialists oversee wage and nonwage programs that an organization provides to its employees in..."


                             ooh_title                                                                                             description_raw                                                                                            description
53                   Childcare Workers               Childcare workers attend to children&rsquo;s needs while helping to foster early development.                Childcare workers attend to children’s needs while helping to foster early development.
142                 Genetic Counselors  Genetic counselors assess clients&apos; risk for a variety of inherited conditions, such as birth defects.  Genetic counselors assess clients' risk for a variety of inherited conditions, such as birth defects.
312  Surveying and Mapping Technicians                   Surveying and mapping technicians collect data and make maps of the Earth&apos; surface.                     Surveying and mapping technicians collect data and make maps of the Earth' surface.


**Profile findings**
- Descriptions run 42 to 347 characters (median 113); 315 of 343 are one sentence, 25 are two (one per job, e.g. *Software Developers, Quality Assurance Analysts, and Testers*), and 2 have no sentence end the regex counts.
- Three still carry HTML entities or a non-breaking space (`&rsquo;`, `&apos;`; *Childcare Workers*, *Genetic Counselors*, *Surveying and Mapping Technicians*, none on the job page). Staging decodes them. The third also has a BLS typo ("the Earth&apos; surface"), kept as published.
- All 343 profiles carry the same publish date, 2026-08-27, one day after the file's "last updated" stamp.

## 4. Coverage of the job page's jobs

The job page covers 225 jobs (`mart.job_profile`); the 23 "All other" groups are counted apart, as in the build plan §2, leaving 202 named jobs.

In [5]:
jp = q("select soc_code, soc_title, soc_level, is_all_other from mart.job_profile")
on = q("select soc_code, description as onet_description from staging.stg_onet__occupation where onet_soc_code like '%%.00'")
cov = (jp.merge(socs, on="soc_code", how="left").merge(prof, on="ooh_code", how="left")
         .merge(table, on=["ooh_code", "soc_code"], how="left").merge(on, on="soc_code", how="left"))
cov["onet_first"] = cov.onet_description.map(lambda s: first_sentence(s) if isinstance(s, str) else None)
named = cov[~cov.is_all_other].copy()
assert len(named) == 202 and named.soc_code.is_unique
named["coverage"] = named.n_soc.map(lambda n: "no OOH profile" if pd.isna(n) else "own profile" if n == 1 else "shared profile")
display(named.coverage.value_counts().to_frame("named jobs"))
print("'All other' groups with a profile:", cov[cov.is_all_other].ooh_code.notna().sum(), "of", cov.is_all_other.sum())
print("non-detailed named codes:", named[named.soc_level != "detailed"][["soc_code", "soc_title", "soc_level", "ooh_title"]].to_dict("records"))

,named jobs
coverage,
own profile,126
shared profile,63
no OOH profile,13


'All other' groups with a profile: 3 of 23
non-detailed named codes: [{'soc_code': '13-1020', 'soc_title': 'Buyers and Purchasing Agents', 'soc_level': 'broad', 'ooh_title': 'Purchasing Managers, Buyers, and Purchasing Agents'}, {'soc_code': '13-2020', 'soc_title': 'Property Appraisers and Assessors', 'soc_level': 'broad', 'ooh_title': 'Property Appraisers and Assessors'}, {'soc_code': '25-2052', 'soc_title': 'Special education teachers, kindergarten and elementary school', 'soc_level': 'bls_aggregate', 'ooh_title': 'Special Education Teachers'}, {'soc_code': '29-2010', 'soc_title': 'Clinical Laboratory Technologists and Technicians', 'soc_level': 'broad', 'ooh_title': 'Clinical Laboratory Technologists and Technicians'}]


**Coverage findings**
- 189 of the 202 named jobs get a BLS sentence: 126 from a profile of their own, 63 through a shared profile. 13 have no profile. These are the same counts as session i's research.
- The four non-detailed named codes are all covered (`13-1020`, `13-2020`, `29-2010` broad; `25-2052` the BLS group code for special education teachers).
- Only 3 of the 23 "All other" groups have a profile. Their descriptions aren't decided here: the job page's coverage counts exclude them (build plan §2).

Jobs with their own profile, where BLS's title differs from the SOC title (read to confirm the sentence still fits; a profile with one SOC code is written for that job):

In [6]:
own = named[named.coverage == "own profile"]
norm = lambda s: re.sub(r"[^a-z]", "", s.lower())
display(own[own.ooh_title.map(norm) != own.soc_title.map(norm)][["soc_code", "soc_title", "ooh_title", "description"]])

,soc_code,soc_title,ooh_title,description
17,11-9031,"Education and Childcare Administrators, Preschool and Daycare",Preschool and Childcare Center Directors,"Preschool and childcare center directors supervise and lead their staffs, design program plans, oversee daily activities, and prepare bu..."
20,11-9072,"Entertainment and Recreation Managers, Except Gambling",Entertainment and Recreation Managers,"Entertainment and recreation managers plan, direct, and coordinate activities and operations related to fitness and leisure."
39,13-1161,Market Research Analysts and Marketing Specialists,Market Research Analysts,"Market research analysts study consumer preferences, business conditions, and other factors to assess potential sales of a product or se..."
71,17-1011,"Architects, Except Landscape and Naval",Architects,"Architects plan and design houses, factories, office buildings, and other structures."
83,17-2111,"Health and Safety Engineers, Except Mining Safety Engineers and Inspectors",Health and Safety Engineers,"Health and safety engineers apply their knowledge of industrial processes, mechanics, psychology, and other disciplines to promote works..."
108,19-2021,Atmospheric and Space Scientists,"Atmospheric Scientists, Including Meteorologists","Atmospheric scientists study, report on, and forecast the weather and climate."
111,19-2041,"Environmental Scientists and Specialists, Including Health",Environmental Scientists and Specialists,Environmental scientists and specialists conduct research or investigations to protect the environment or human health.
112,19-2042,"Geoscientists, Except Hydrologists and Geographers",Geoscientists,Geoscientists study the physical aspects of the Earth.
119,19-4042,"Environmental Science and Protection Technicians, Including Health",Environmental Science and Protection Technicians,"Environmental science and protection technicians monitor the environment and investigate sources of pollution and contamination, includi..."
138,25-2011,"Preschool Teachers, Except Special Education",Preschool Teachers,Preschool teachers educate and care for children younger than age 5 who have not yet entered kindergarten.


**Read:** in all 15, BLS's title is a shorter or more common name for the same job ("High School Teachers" for 25-2031, "Industrial Designers" for 27-1021), and the sentence fits. No override needed.

The 13 named jobs with no OOH profile fall back to the first sentence of O\*NET's `.00` definition (O\*NET 31.0, CC BY 4.0):

In [7]:
none = named[named.coverage == "no OOH profile"][["soc_code", "soc_title", "onet_first"]].assign(chars=lambda x: x.onet_first.str.len())
display(none)
assert none.onet_first.notna().all()

,soc_code,soc_title,onet_first,chars
26,13-1011,"Agents and Business Managers of Artists, Performers, and Athletes","Represent and promote artists, performers, and athletes in dealings with current or prospective employers.",106
44,13-2041,Credit Analysts,Analyze credit data and financial statements of individuals or firms to determine the degree of risk involved in extending credit or len...,147
121,19-4061,Social Science Research Assistants,"Assist social scientists in laboratory, survey, and other social science research.",82
122,19-4071,Forest and Conservation Technicians,"Provide technical assistance regarding the conservation of soil, water, forests, or related natural resources.",110
131,21-2011,Clergy,Conduct religious worship and perform other spiritual functions associated with beliefs and practices of religious faith or denomination.,137
132,21-2021,"Directors, Religious Activities and Education",Coordinate or design programs and conduct outreach to promote the religious education or activities of a denominational group.,126
150,25-3031,"Substitute Teachers, Short-Term","Teach students on a short-term basis as a temporary replacement for a regular classroom teacher, typically using the regular teacher's l...",147
153,25-9044,"Teaching Assistants, Postsecondary","Assist faculty or other instructional staff in postsecondary institutions by performing instructional support activities, such as develo...",258
192,29-2051,Dietetic Technicians,"Assist in the provision of food service and nutritional programs, under the supervision of a dietitian.",103
203,43-9081,Proofreaders and Copy Markers,"Read transcript or proof type setup to detect and mark for correction any grammatical, typographical, or compositional errors.",126


**Fallback findings:** all 13 have an O\*NET `.00` definition, so the fallback is complete. They read in the imperative ("Represent and promote artists…"), as planned (R2 option B). One runs 258 characters (25-9044 Teaching Assistants, Postsecondary), over the 256 the build plan's test assumed (§6). The real limit is the layout rule, "at most 2 lines at 12pt" (R2), so step 4 measures it with Windows font metrics, and the dbt test checks the job-page rows at 260.

## 5. Hand check: the 63 named jobs that share a profile

**The test:** read the sentence directly under the job's own title, as the page shows it. It passes if it names this job or BLS's group for it and everything it says is true of this job. If it fails, the replacement is still BLS text, in this order:
1. **`profile_part`**: the profile's description has one sentence per job (e.g. "Software developers design… Software quality assurance analysts and testers identify…"); keep the sentence about this job.
2. **`bls_type`**: the first sentence of the "types of" paragraph in *What They Do* that names this job.
3. **`onet`**: the O\*NET first sentence (needed for none of the 63).

Below, each profile with its sentence, then for every job the candidates: BLS's projections-table title for it, the matching "types of" sentence (matched on that title or the SOC title, ignoring case and punctuation), and the O\*NET first sentence.

In [8]:
sh = named[named.coverage == "shared profile"].copy()
def type_for(r):
    keys = {norm(x) for x in [r.table_title, r.soc_title] if isinstance(x, str)}
    hit = types[(types.ooh_code == r.ooh_code) & types.type_name.map(norm).isin(keys)]
    return hit.type_sentence.iloc[0] if len(hit) else None
sh["type_sentence"] = sh.apply(type_for, axis=1)
print(f"{len(sh)} jobs in {sh.ooh_code.nunique()} profiles; a 'types of' sentence matches by name for {sh.type_sentence.notna().sum()}")
for t, g in sh.sort_values(["ooh_title", "soc_code"]).groupby("ooh_title", sort=False):
    md(f"**{t}** ({int(g.n_soc.iloc[0])} SOC codes in the profile) — *{g.description.iloc[0]}*  \n"
       + "  \n".join(f"`{r.soc_code}` {r.soc_title} · type: {r.type_sentence or '—'} · O\\*NET: {r.onet_first or '—'}"
                     for r in g.itertuples()))

63 jobs in 36 profiles; a 'types of' sentence matches by name for 17

**Administrative Services and Facilities Managers** (2 SOC codes in the profile) — *Administrative services and facilities managers plan, direct, and coordinate activities that help an organization run efficiently.*  
`11-3012` Administrative Services Managers · type: nan · O\*NET: Plan, direct, or coordinate one or more administrative services of an organization, such as records and information management, mail distribution, and other office support services.  
`11-3013` Facilities Managers · type: nan · O\*NET: Plan, direct, or coordinate operations and functionalities of facilities and buildings.

**Advertising, Promotions, and Marketing Managers** (2 SOC codes in the profile) — *Advertising, promotions, and marketing managers plan programs to generate interest in products or services.*  
`11-2011` Advertising and Promotions Managers · type: nan · O\*NET: Plan, direct, or coordinate advertising policies and programs or produce collateral materials, such as posters, contests, coupons, or giveaways, to create extra interest in the purchase of a product or service for a department, an entire organization, or on an account basis.  
`11-2021` Marketing Managers · type: Marketing managers estimate demand and identify potential markets for products and services that an organization and its competitors offer. · O\*NET: Plan, direct, or coordinate marketing policies and programs, such as determining the demand for products and services offered by a firm and its competitors, and identify potential customers.

**Agricultural and Food Science Technicians** (2 SOC codes in the profile) — *Agricultural and food science technicians help scientists with testing and research related to plants, animals, fibers, and food and beverage products.*  
`19-4012` Agricultural Technicians · type: nan · O\*NET: Work with agricultural scientists in plant, fiber, and animal research, or assist with animal breeding and nutrition.  
`19-4013` Food Science Technicians · type: nan · O\*NET: Work with food scientists or technologists to perform standardized qualitative and quantitative tests to determine physical or chemical properties of food or beverage products.

**Agricultural and Food Scientists** (3 SOC codes in the profile) — *Agricultural and food scientists research ways to improve the efficiency, quality, and safety of agricultural and food production establishments.*  
`19-1011` Animal Scientists · type: Animal scientists typically conduct research on livestock. · O\*NET: Conduct research in the genetics, nutrition, reproduction, growth, and development of domestic farm animals.  
`19-1012` Food Scientists and Technologists · type: Food scientists and technologists study the basic elements of food. · O\*NET: Use chemistry, microbiology, engineering, and other sciences to study the principles underlying the processing and deterioration of foods; analyze food content to determine levels of vitamins, fat, sugar, and protein; discover new food sources; research ways to make processed foods safe, palatable, and healthful; and apply food science knowledge to determine best ways to process, package, preserve, store, and distribute food.  
`19-1013` Soil and Plant Scientists · type: nan · O\*NET: Conduct research in breeding, physiology, production, yield, and management of crops and agricultural plants or trees, shrubs, and nursery stock, their growth in soils, and control of pests; or study the chemical, physical, biological, and mineralogical composition of soils as they relate to plant or crop growth.

...

**Special Education Teachers** (5 SOC codes in the profile) — *Special education teachers work with students who have a wide range of learning, mental, emotional, and physical disabilities.*  
`25-2051` Special Education Teachers, Preschool · type: nan · O\*NET: Teach academic, social, and life skills to preschool-aged students with learning, emotional, or physical disabilities.  
`25-2052` Special education teachers, kindergarten and elementary school · type: nan · O\*NET: nan  
`25-2057` Special Education Teachers, Middle School · type: nan · O\*NET: Teach academic, social, and life skills to middle school students with learning, emotional, or physical disabilities.  
`25-2058` Special Education Teachers, Secondary School · type: nan · O\*NET: Teach academic, social, and life skills to secondary school students with learning, emotional, or physical disabilities.

**Telecommunications Technicians** (3 SOC codes in the profile) — *Telecommunications technicians install, maintain, and repair radio, internet, and other telecommunications infrastructure.*  
`49-2021` Radio, Cellular, and Tower Equipment Installers and Repairers · type: Radio, cellular, and tower equipment installers and repairers install, repair, or maintain radio transmitting, broadcasting, and receiving equipment and two-way radio communications systems. · O\*NET: Repair, install, or maintain mobile or stationary radio transmitting, broadcasting, and receiving equipment, and two-way radio communications systems used in cellular telecommunications, mobile broadband, ship-to-shore, aircraft-to-ground communications, and radio equipment in service and emergency vehicles.

**Top Executives** (2 SOC codes in the profile) — *Top executives plan strategies and policies to ensure that an organization meets its goals.*  
`11-1011` Chief Executives · type: nan · O\*NET: Determine and formulate policies and provide overall direction of companies or private and public sector organizations within guidelines set up by a board of directors or similar governing body.  
`11-1021` General and Operations Managers · type: General and operations managers oversee activities that are too diverse to be classified into one area of management or administration. · O\*NET: Plan, direct, or coordinate the operations of public or private sector organizations, overseeing multiple departments or locations.

**Web Developers and Digital Designers** (2 SOC codes in the profile) — *Web developers design and create websites.*  
`15-1254` Web Developers · type: nan · O\*NET: Develop and implement websites, web applications, application databases, and interactive web interfaces.  
`15-1255` Web and Digital Interface Designers · type: Web and digital interface designers are responsible for creating the look and feel of a website or interface with regard to photos, color, font type and size, graphics, and layout. · O\*NET: Design digital user interfaces or websites.

**Wholesale and Manufacturing Sales Representatives** (2 SOC codes in the profile) — *Wholesale and manufacturing sales representatives sell goods for wholesalers or manufacturers to businesses, government agencies, and other organizations.*  
`41-4011` Sales Representatives, Wholesale and Manufacturing, Technical and Scientific Products · type: nan · O\*NET: Sell goods for wholesalers or manufacturers where technical or scientific knowledge is required in such areas as biology, engineering, chemistry, and electronics, normally obtained from at least 2 years of postsecondary education.

**The verdicts** (seed `ref/ref_job_description_override.csv`, one row per job; `description` is filled only when the BLS sentence is replaced). Checks: the seed lists exactly these 63 jobs, and every replacement is BLS's own text, word for word: a sentence of this profile's description (`profile_part`) or this profile's "types of" sentence (`bls_type`).

In [9]:
seed = pd.read_csv(PROJECT / "ref" / "ref_job_description_override.csv", dtype=str, keep_default_na=False)
assert set(seed.soc_code) == set(sh.soc_code) and seed.soc_code.is_unique
assert set(seed.verdict) <= {"keep", "profile_part", "bls_type", "onet"}
chk = seed.merge(sh[["soc_code", "description"]].rename(columns={"description": "ooh_description"}), on="soc_code")
chk["verbatim"] = check_description_overrides(seed, ooh).values
assert chk.verbatim.all(), chk[~chk.verbatim]
display(seed.verdict.value_counts().to_frame("jobs"))
display(chk[chk.verdict != "keep"][["soc_code", "soc_title", "verdict", "ooh_description", "description"]])
md("**One borderline keep:** " + seed.loc[seed.reason.str.startswith("borderline"), ["soc_code", "soc_title", "reason"]].to_string(index=False, header=False))

,jobs
verdict,
keep,47
profile_part,9
bls_type,7


,soc_code,soc_title,verdict,ooh_description,description
4,11-2032,Public Relations Managers,profile_part,Public relations managers direct the creation of materials that will enhance the public image of their employer or client. Fundraising m...,Public relations managers direct the creation of materials that will enhance the public image of their employer or client.
5,11-2033,Fundraising Managers,profile_part,Public relations managers direct the creation of materials that will enhance the public image of their employer or client. Fundraising m...,Fundraising managers coordinate campaigns that bring in donations for their organization.
8,11-3061,Purchasing Managers,profile_part,Buyers and purchasing agents buy products and services for organizations. Purchasing managers oversee the work of buyers and purchasing ...,Purchasing managers oversee the work of buyers and purchasing agents.
9,11-9171,Funeral Home Managers,bls_type,Funeral service workers organize and manage the details of a ceremony honoring a deceased person.,Funeral home managers oversee the general operations of a funeral home business.
10,13-1020,Buyers and Purchasing Agents,profile_part,Buyers and purchasing agents buy products and services for organizations. Purchasing managers oversee the work of buyers and purchasing ...,Buyers and purchasing agents buy products and services for organizations.
12,13-2054,Financial Risk Specialists,bls_type,"Financial analysts research and evaluate financial data, forecast future trends, and prepare reports containing recommendations for busi...","Financial risk specialists, also called financial risk analysts, assess and measure exposure to credit and market risk."
16,15-1252,Software Developers,profile_part,Software developers design computer applications or programs. Software quality assurance analysts and testers identify problems with app...,Software developers design computer applications or programs.
17,15-1253,Software Quality Assurance Analysts and Testers,profile_part,Software developers design computer applications or programs. Software quality assurance analysts and testers identify problems with app...,Software quality assurance analysts and testers identify problems with applications or programs and report defects.
19,15-1255,Web and Digital Interface Designers,bls_type,Web developers design and create websites.,"Web and digital interface designers are responsible for creating the look and feel of a website or interface with regard to photos, colo..."
47,25-4013,Museum Technicians and Conservators,profile_part,Archivists and curators oversee collections of historic items or artwork. Museum technicians and conservators prepare or restore items i...,Museum technicians and conservators prepare or restore items in museum collections.


**One borderline keep:** 11-1021 General and Operations Managers borderline: 'Top executives plan strategies and policies…' is BLS's group for this job and true of it; the job's own 'types of' sentence ('…activities that are too diverse to be classified into one area…') reads worse

**Hand-check findings**
- **47 keep the shared sentence**: it names the job or BLS's group for it and is true of the job (*Drafters use software to convert the designs of engineers and architects into technical drawings* under Mechanical Drafters).
- 9 take one half of a two-job description (Software Developers / QA Analysts; Radiologic / MRI Technologists; PR / Fundraising Managers; Purchasing Managers / Buyers; Museum Technicians).
- 7 take BLS's "types of" sentence because the shared one names another job or describes another job's work: Fish and Game Wardens (police), Financial Risk Specialists (financial analysts), HR Assistants (information clerks), Web and Digital Interface Designers (web developers), Funeral Home Managers (arranging a ceremony), Occupational Therapy Assistants (the aides' routine tasks), Broadcast Announcers and Radio DJs (wedding DJs).
- **None needs O\*NET.** Every replacement is BLS's own wording, checked word for word against the file above.
- One borderline keep to look at: 11-1021 General and Operations Managers keeps "Top executives plan strategies and policies…".

## 6. The description each named job gets, and the tie-out

Rule for `dim_job.job_description` (step 2): the seed's text when it replaces; otherwise the OOH sentence; otherwise the O\*NET first sentence. `description_source` = `bls_ooh` or `onet`.

In [10]:
fin = named.merge(seed[["soc_code", "verdict", "description"]].rename(columns={"description": "override"}), on="soc_code", how="left")
fin["job_description"] = fin.override.where(fin.override.fillna("") != "", fin.description).fillna(fin.onet_first)
fin["description_source"] = fin.description.notna().map({True: "bls_ooh", False: "onet"})
fin["chars"] = fin.job_description.str.len()
assert fin.job_description.notna().all()
fin["how"] = fin.verdict.fillna(fin.coverage)
display(fin.groupby(["description_source", "how"]).size().to_frame("jobs"))
print("length over the 202:", fin.chars.describe().round(0).to_dict(), "| over 256 characters:",
      fin.loc[fin.chars > 256, ["soc_code", "chars"]].to_dict("records"))
tie = fin.set_index("soc_code").loc[["13-2011", "25-2021", "33-3031", "15-1252"], ["soc_title", "job_description", "description_source"]]
display(tie)
assert tie.loc["13-2011", "job_description"] == "Accountants and auditors prepare and examine financial records."  # build plan §5

jobs
description_source how                 
bls_ooh            bls_type           7
                   keep              47
                   own profile      126
                   profile_part       9
onet               no OOH profile    13

length over the 202: {'count': 202.0, 'mean': 119.0, 'std': 35.0, 'min': 42.0, '25%': 96.0, '50%': 116.0, '75%': 139.0, 'max': 258.0} | over 256 characters: [{'soc_code': '25-9044', 'chars': 258}]


,soc_title,job_description,description_source
soc_code,,,
13-2011,Accountants and Auditors,Accountants and auditors prepare and examine financial records.,bls_ooh
25-2021,"Elementary School Teachers, Except Special Education",Kindergarten and elementary school teachers instruct young students in basic subjects in order to prepare them for future schooling.,bls_ooh
33-3031,Fish and Game Wardens,"Fish and game wardens enforce fishing, hunting, and boating laws.",bls_ooh
15-1252,Software Developers,Software developers design computer applications or programs.,bls_ooh


**Result ** All 202 named jobs get a description: 189 from BLS (126 own profile, 47 shared and kept, 9 one half, 7 "types of") and **13 from O\*NET**. 13-2011 matches the mockup and the build plan §5 word for word, and 33-3031 now describes fish and game wardens.

## 7. Findings and the proposed staging table

See the printed results above; the conclusions are in the source card `bls_ooh.md`. Staged by notebook 36 as `stg_bls__ooh_description`: one row per SOC code in the file (every profile, not only the job page's jobs), key `soc_code`; columns `soc_code`, `ooh_code`, `ooh_title`, `description`, `n_soc_in_profile`, `publish_date`, `source_file`. The hand-check verdicts stay in the seed, and the "types of" text isn't staged: the seed carries the replacement sentence, and notebook 36 re-checks it word for word against the file.